In [ ]:
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"

!pip install --no-deps "xformers<0.0.27" "trl<0.9.0" peft accelerate bitsandbytes

  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-k856uh2g/unsloth_173637f4eac649a28d4a0c90ccec017c
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-k856uh2g/unsloth_173637f4eac649a28d4a0c90ccec017c
  Resolved https://github.com/unslothai/unsloth.git to commit e280b0bebc0b9aacb12154500a99c9fa3e69e345
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 11.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 18.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 22.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 395.2/395.2 kB 29.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 80.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 182.6/182.6 kB 17.4 MB/s eta 0:00:00


KeyboardInterrupt: 

In [ ]:
!pip install faiss-cpu nltk lxml wordninja rank-bm25
!pip install wikipedia-api

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 541.6/541.6 kB 9.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 65.9 MB/s eta 0:00:00
  Created wheel for wordninja: filename=wordninja-2.0.0-py3-none-any.whl size=541530 sha256=0f651df3ee1e48901590bc7e6176ddc3499256c7765908bd9eda8fc20b0edb88
  Stored in directory: /root/.cache/pip/wheels/6e/31/92/f12667e4dd102e546832a02f41feca39ae916889006517e595
Successfully built wordninja
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 2.7 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incomp

In [ ]:
#--------------Import Unsloth----------------------------
# from unsloth import FastLanguageModel
# from unsloth import FastModel
# import torch

#--------------Transformers Import------------------------------------
import wandb
from sentence_transformers import SentenceTransformer
from sentence_transformers import CrossEncoder
from transformers import AutoModelForSequenceClassification, AutoTokenizer, pipeline
from google.colab import drive
from datasets import Dataset
from rank_bm25 import BM25Okapi
import faiss

# -------------------------------------------------------
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report
import pandas as pd
import tensorflow as tf
from tensorflow.keras.models import load_model
from collections import Counter
from tensorflow.keras.utils import register_keras_serializable
from tensorflow.keras.optimizers import AdamW
import nltk
nltk.download("punkt_tab")
nltk.download("stopwords")

# --------------------------------------------------------
import requests
from bs4 import BeautifulSoup
import lxml
import json
from collections import defaultdict
import re
import os
import wordninja
from IPython.display import clear_output
import warnings
warnings.filterwarnings("ignore")

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


In [ ]:
drive.mount('/content/drive')
model_path = "/content/drive/MyDrive/Audios/toxicity_quality_bert_model"
tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForSequenceClassification.from_pretrained(model_path)

MessageError: Error: credential propagation was unsuccessful

In [ ]:
Genmodel, Phitokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Phi-3-mini-4k-instruct-bnb-4bit",
    max_seq_length = 2048,
    load_in_4bit = True,
)
FastLanguageModel.for_inference(Genmodel)

In [ ]:
def toxicity_check(text : str) -> dict:
  device = torch.device("cpu")
  model.to(device)
  model.eval()
  inputs = tokenizer(
      text,
      return_tensors="pt",
      truncation=True,
      max_length=128,
      padding="max_length"
  ).to(device)

  with torch.no_grad():
    outputs = model(**inputs)

  probabilities = torch.sigmoid(outputs.logits).squeeze().tolist()
  categories = [
      "Toxic",
      "Threat",
      "Profanity",
      "Incomplete (Reproduce)",
      "Bad Quality (Remake)"
  ]

  prompt_info = {}
  for name, prob in zip(categories, probabilities):
    prompt_info[name] = prob

  return prompt_info

def prompt_fix(text : str) -> str:
    messages = [
        {
            "role": "system",
            "content": (
                "You are a Prompt Expander. Your ONLY job is to rewrite the input into a detailed RAG prompt. "
                "NEVER answer the question. NEVER provide facts about the animal. "
                "ONLY output a new instruction that starts with 'Act as an expert...'"
            )
        },
        {
            "role": "user",
            "content": f"Rewrite this into a detailed RAG instruction, do not answer it: [{text}]\n\nStrict Output:"
        },
    ]


    inputs = Phitokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True, return_tensors="pt").to("cuda")

    # Using do_sample=False makes the model much more likely to follow strict formatting
    outputs = Genmodel.generate(
        input_ids=inputs,
        max_new_tokens=300,
        temperature=0.1,
        do_sample=False
    )

    # We decode and extract ONLY what comes after the assistant tag
    full_text = Phitokenizer.decode(outputs[0], skip_special_tokens=True)

    # This split ensures we only get the model's new generation
    if "assistant" in full_text:
        return full_text.split("assistant")[-1].strip()

    exact_prompt = nltk.sent_tokenize(full_text.strip())
    sys_prompt = nltk.sent_tokenize(messages[0]['content'])
    final = "".join([i for i in exact_prompt if i not in sys_prompt])
    final = final.split("Strict Output:")[-1].strip()
    return final

def prompt_optimizer(text: str) -> str:
    prompt_info = toxicity_check(text)

    is_toxic = (prompt_info['Toxic'] > 0.5 and prompt_info['Profanity'] > 0.5) or prompt_info['Threat'] > 0.5
    is_low_quality = prompt_info['Incomplete (Reproduce)'] > 0.5 or prompt_info['Bad Quality (Remake)'] > 0.5

    if is_toxic :
      return "Sorry, we can help you with that !!"

    # if is_low_quality:
    #     return prompt_fix(text)
    return prompt_fix(text)

In [ ]:
test_text = ""
print(prompt_optimizer(test_text))

test_text_3 = "Tell me about hyrax ??"
print(prompt_optimizer(test_text_3))

test_text_4 = "Abuse me in different languages"
print(prompt_optimizer(test_text_4))

test_text_5 = "Go fuck yourself"
print(prompt_optimizer(test_text_5))

test_text_5 = "Tell me about slender loris. Why is it so shy in nature ??"
print(prompt_optimizer(test_text_5))

Act as an expert in the field of zoology, focusing on the study of animal behavior and communication.Your task is to create a comprehensive RAG (Recurrent Attention Generative) prompt that encapsulatively summarizes the intricate relationship between animal behavior and communication.This prompt should be designed to guide a language model in generating a detailed analysis of how various species utilize communication as a pivotal aspect of their behavior.The prompt must include key elements such as the importance of communication in social animals, the role of communication in mating rituals, and the impact of environmental factors on communication methods.Ensure that the prompt is structured to facilitate a deep dive into the complexities of animal behavior and communication, without directly referencing specific facts about any particular animal.
Act as an expert zoologist and provide a comprehensive overview of the hyrax, a small, herbivorous mammal belonging to the family Hyracoide

In [ ]:
@register_keras_serializable(package="Custom", name="WarmUpLearningRate")
class WarmUpLearningRate(tf.keras.optimizers.schedules.LearningRateSchedule):
    def __init__(self, initial_lr, warmup_steps, decay_schedule_fn):
        super().__init__()
        self.initial_lr = initial_lr
        self.warmup_steps = warmup_steps
        self.decay_schedule_fn = decay_schedule_fn
    def __call__(self, step):
        step = tf.cast(step, tf.float32)
        warmup_lr = self.initial_lr * (step / tf.cast(self.warmup_steps, tf.float32))
        return tf.cond(
            step < self.warmup_steps,
            lambda: warmup_lr,
            lambda: self.decay_schedule_fn(step - self.warmup_steps)
        )
    def get_config(self):
        return {
            "initial_lr": self.initial_lr,
            "warmup_steps": self.warmup_steps,
            "decay_schedule_fn": tf.keras.optimizers.schedules.serialize(self.decay_schedule_fn),
        }
    @classmethod
    def from_config(cls, config):
        config["decay_schedule_fn"] = tf.keras.optimizers.schedules.deserialize(config["decay_schedule_fn"])
        return cls(**config)

early_stop = tf.keras.callbacks.EarlyStopping(
    monitor = 'val_loss',
    patience = 10,
    restore_best_weights= True
)
decay_schedule = tf.keras.optimizers.schedules.PolynomialDecay(
    initial_learning_rate=0.001,
    decay_steps=10000,
    end_learning_rate=1e-5,
    power=1.05
)
warmup_lr = WarmUpLearningRate(
    initial_lr=0.001,
    warmup_steps=1000,
    decay_schedule_fn=decay_schedule
)
optimizer = AdamW(
    learning_rate=warmup_lr,
    weight_decay=1e-4
)
embedding_model = SentenceTransformer("multi-qa-MiniLM-L6-cos-v1")  # or "all-mpnet-base-v2"
EMBED_DIM = embedding_model.get_sentence_embedding_dimension()
print("embed dim:", EMBED_DIM)
ANN = load_model("/content/NER_Model_1.keras")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

embed dim: 384


In [ ]:
def sentence_to_embedding(sentence, model, embedding_dim=384):
    words = nltk.word_tokenize(sentence)
    stop = nltk.corpus.stopwords.words('english')
    words = [word for word in words if word.lower() not in stop and word.isalnum()]
    if not words:
        return np.zeros((1, embedding_dim))
    embeddings = model.encode(words)
    if embeddings.ndim == 1:
        embeddings = embeddings[np.newaxis, :]
    mean_embedding = embeddings.mean(axis=0)
    return mean_embedding.reshape(1, -1)

LABELS = ['Adaptation', 'Behavior', 'Diet', 'Feature', 'Genus',
          'Gpe', 'Habitat', 'Roleinecosystem', 'Threat', 'Timeperiod']

def predict_label(sentence, model, ANN):
    embedding = sentence_to_embedding(sentence, model)
    pred = ANN.predict(embedding)
    pred_label_idx = np.argmax(pred, axis=1)
    return LABELS[pred_label_idx[0]]

In [ ]:
def format_wiki_name(animal_name):

    words = wordninja.split(animal_name.strip())

    words = [w.capitalize() for w in words]

    return "_".join(words)

def search_wikipedia(animal_name):

    search_url = "https://en.wikipedia.org/w/api.php"

    headers = {
        "User-Agent": "Mozilla/5.0"
    }

    params = {
        "action": "query",
        "list": "search",
        "srsearch": animal_name,
        "format": "json"
    }

    response = requests.get(search_url, params=params, headers=headers)

    if response.status_code != 200:
        return None

    data = response.json()

    if data["query"]["search"]:
        title = data["query"]["search"][0]["title"]
        return title.replace(" ", "_")

    return None

def extract_infobox_facts(soup, animal, scientific_name):

    facts = []

    infobox = soup.find("table", {"class": "infobox"})

    if not infobox:
        return facts

    rows = infobox.find_all("tr")

    for row in rows:

        th = row.find("th")
        td = row.find("td")

        if th and td:

            key = th.get_text(" ", strip=True)
            val = td.get_text(" ", strip=True)

            val = re.sub(r"\[[^\]]*\]", "", val)

            if len(val) < 200:

                facts.append({
                    "animal": animal,
                    "scientific_name": scientific_name,
                    "section": "Infobox",
                    "attribute": key,
                    "source": "Wikipedia",
                    "text": f"{animal} {key}: {val}"
                })

    return facts

def extract_sections(soup):

    sections = {}

    current_section = "Introduction"

    for tag in soup.find_all(["h2", "h3", "p"]):

        if tag.name in ["h2", "h3"]:

            title = tag.get_text()
            title = re.sub(r"\[.*?\]", "", title)
            title = title.replace("edit", "").strip()

            current_section = title

        elif tag.name == "p":

            text = tag.get_text().strip()

            if len(text) > 80:
                sections.setdefault(current_section, []).append(text)

    return sections

def extract_classification(soup, animal, scientific_name):

    facts = []

    infobox = soup.find("table", {"class": "infobox"})

    if not infobox:
        return facts

    rows = infobox.find_all("tr")

    for row in rows:

        header = row.find("th")
        value = row.find("td")

        if header and value:

            attribute = header.get_text(" ", strip=True)
            val = value.get_text(" ", strip=True)

            # remove citations like [1]
            val = re.sub(r"\[\d+\]", "", val)

            if attribute in [
                "Kingdom", "Phylum", "Class",
                "Order", "Family", "Genus",
                "Species", "Tribe"
            ]:

                facts.append({
                    "animal": animal,
                    "scientific_name": scientific_name,
                    "section": "Scientific classification",
                    "attribute": attribute,
                    "value": val,
                    "source": "Wikipedia",
                    "text": f"{animal} ({scientific_name}) {attribute}: {val}"
                })

    return facts

def section_to_facts(animal, scientific_name, sections):

    facts = []

    for section, paragraphs in sections.items():

        section_lower = section.lower()

        for para in paragraphs:

            sentences = nltk.sent_tokenize(para)

            for s in sentences:

                s = re.sub(r"\[[^\]]*\]", "", s)
                s = re.sub(r"\s+", " ", s).strip()

                if 40 < len(s) < 250:
                    label = predict_label(s, embedding_model, ANN)

                    # Detect conservation / endangered sections
                    if re.search(r"\b(endangered|critically endangered|vulnerable|threatened)\b", s.lower()):
                      label = "Endangered"

                    facts.append({
                        "animal": animal,
                        "scientific_name": scientific_name,
                        "section": section,
                        "label": label,
                        "source": "Wikipedia",
                        "text": f"{animal} ({scientific_name}) {section}: {s}"
                    })

    return facts

def scrape_wikipedia(animal_name):

    formatted_name = format_wiki_name(animal_name)

    url = f"https://en.wikipedia.org/wiki/{formatted_name}"

    headers = {
        "User-Agent": "Mozilla/5.0"
    }

    print("Fetching:", url)

    response = requests.get(url, headers=headers, allow_redirects=True)

    print("Final page:", response.url)

    if response.status_code != 200:
        print("⚠️ Page not found")
        return []

    soup = BeautifulSoup(response.text, "html.parser")

    # -------- Scientific Name --------
    scientific_name = "Unknown"

    infobox = soup.find("table", {"class": "infobox"})

    if infobox:

        species_row = infobox.find("th", string="Species")

        if species_row:
            td = species_row.find_next("td")

            if td:
                scientific_name = td.get_text(" ", strip=True)

        else:
            italic = infobox.find("i")
            if italic:
                scientific_name = italic.text

    # -------- Classification --------
    classification_facts = extract_classification(
        soup, animal_name, scientific_name
    )

    # -------- Sections --------
    sections = extract_sections(soup)

    section_facts = section_to_facts(
        animal_name,
        scientific_name,
        sections
    )

    infobox_facts = extract_infobox_facts(
        soup, animal_name, scientific_name
    )

    # combine both
    facts = classification_facts + section_facts + infobox_facts

    return facts

def save_dataset(facts, filename="animal_dataset.json"):

    existing_data = []

    # If file already exists, load old data
    if os.path.exists(filename):
        with open(filename, "r", encoding="utf-8") as f:
            try:
                existing_data = json.load(f)
            except json.JSONDecodeError:
                existing_data = []

    # Add new facts
    existing_data.extend(facts)

    # Save combined dataset
    with open(filename, "w", encoding="utf-8") as f:
        json.dump(existing_data, f, indent=4, ensure_ascii=False)

    print(f"Added {len(facts)} new facts. Total facts: {len(existing_data)}")


In [ ]:
def main():
    lt = ['Lion-tailed macaque','Pygmy Marmoset','Slender loris','African elephant','Lion','Tiger','Cheetah','Chamelion','Giraffe','Hyena','Spotted hyena','Striped hyena']
    for i in lt:
      clear_output(wait=True)
      animal_name = i

      facts = scrape_wikipedia(animal_name)

      print("Facts collected:", len(facts))

      save_dataset(facts)

      print("Dataset saved!")
main()

Fetching: https://en.wikipedia.org/wiki/Striped_Hyena
Final page: https://en.wikipedia.org/wiki/Striped_Hyena
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 61ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 53ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 55ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 75ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 53ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 54ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 53ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 55ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 57ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 61ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 51ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 53ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 60ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 50ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 53ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 50ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 58ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 52ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 49ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 54ms/step
1/1 ━━━━━━━━━━━━

In [ ]:
def restructure_and_split_for_faiss(data):

    grouped_facts = defaultdict(list)
    final_data = []

    # 1. Group facts by label
    for item in data['facts']:
        grouped_facts[item['label']].append(item['text'])

    # 2. Process each label
    for label, texts in grouped_facts.items():

        # Skip unwanted labels
        if label.lower() in ['feature', 'roleinecosystem']:
            continue

        combined_text = " ".join(texts)

        # 3. Split paragraph into sentences
        sentences = re.split(r'(?<=[.!?]) +', combined_text)

        for sentence in sentences:
            sentence = sentence.strip()

            if len(sentence) > 5:
                chunk = {
                    "subject": data['Name'],
                    "topic": label,
                    "scientific_name" : data['scientific_name'],
                    "content": sentence,
                    "metadata": {
                        "source": data.get('source', 'Unknown'),
                        "is_behavioral": label == 'Behavior'
                    }
                }

                final_data.append(chunk)

    return final_data

In [ ]:
def save(final_dataset):
  filename = "ANIMAL_WIKIPEDIA_DATA.json"

  try:
      # 1. Check if file exists and has content
      if os.path.exists(filename) and os.path.getsize(filename) > 0:
          with open(filename, 'r') as json_file:
              # Load existing data
              current_data = json.load(json_file)
      else:
          # Initialize as an empty list if the file doesn't exist
          current_data = []
      if isinstance(final_dataset, list):
          current_data.extend(final_dataset)
      else:
          current_data.append(final_dataset)

      # 3. Write the combined data back to the file
      with open(filename, 'w') as json_file:
          json.dump(current_data, json_file, indent=4)

      print(f"Data successfully merged and saved to {filename}")

  except (IOError, json.JSONDecodeError) as e:
      print(f"Error handling file: {e}")

In [ ]:
# ========== 4️⃣ MAIN EXECUTION ==========
def collect_data(animal_name):
    #animal_name = input("Enter animal name: ").strip()
    animal_name = animal_name.strip()
    data_1, scientific_name = web1(animal_name)
    # data_2 = web2(animal_name)

    animal_card_1 = build_animal_card(animal_name, scientific_name, data_1, "Wikipedia")
    final_dataset = restructure_and_split_for_faiss(animal_card_1)
    print(final_dataset)
    return save(final_dataset)
lt = ['Tiger','Lion','Tiger Quoll','Pygmy_marmoset','Gorilla','Macaque','Lion-tailed Macaque']
for i in lt:
  clear_output(wait=True)
  collect_data(i)

# # Run
# animal_card_1 = main()

# print("\n🐾 ANIMAL CARD 1 (Wikipedia):")
# print(animal_card_1)

Lion_tailed_macaque
⚠️ Error fetching https://en.wikipedia.org/wiki/Lion_tailed_macaque
[]
Data successfully merged and saved to ANIMAL_WIKIPEDIA_DATA.json


In [ ]:
collect_data('Lion-tailed Macaque')

Lion_tailed_macaque
⚠️ Error fetching https://en.wikipedia.org/wiki/Lion_tailed_macaque
[]
Data successfully merged and saved to ANIMAL_WIKIPEDIA_DATA.json


In [ ]:
with open('/content/ANIMAL_WIKIPEDIA_DATA.json', 'r') as file:
    data = json.load(file)

print("Data loaded successfully:")
print(data)
print(f"Type of data: {type(data)}")

NameError: name 'json' is not defined

In [ ]:
documents = []
metadata = []

for entry in data:

    text = f"{entry['subject']} {entry['topic']} {entry['content']}"

    documents.append(text)
    metadata.append(entry)

In [ ]:
faiss_embedding_model = SentenceTransformer("BAAI/bge-base-en-v1.5")

embeddings = faiss_embedding_model.encode(documents, convert_to_numpy=True)
embeddings = embeddings.astype("float32")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/777 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/366 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(embeddings)

In [ ]:
#Build BM25 Keyword Search
tokenized_docs = [doc.lower().split() for doc in documents]
bm25 = BM25Okapi(tokenized_docs)

In [ ]:

tokenized_docs = [doc.lower().split() for doc in documents]

bm25 = BM25Okapi(tokenized_docs)

In [ ]:
def faiss_search(query, k=10):

    query_embedding = model.encode([query]).astype("float32")

    distances, indices = index.search(query_embedding, k)

    results = [metadata[i] for i in indices[0]]

    return results

In [ ]:
def bm25_search(query, k=10):

    tokens = query.lower().split()

    scores = bm25.get_scores(tokens)

    top_idx = np.argsort(scores)[::-1][:k]

    results = [metadata[i] for i in top_idx]

    return results

In [ ]:
def hybrid_search(query):

    faiss_results = faiss_search(query, 10)

    bm25_results = bm25_search(query, 10)

    combined = faiss_results + bm25_results

    # remove duplicates
    seen = set()
    unique = []

    for item in combined:
        key = item["content"]
        if key not in seen:
            seen.add(key)
            unique.append(item)

    return unique[:10]

In [ ]:
reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

In [ ]:
with open('/content/ANIMAL_WIKIPEDIA_DATA.json', 'r') as file:
    data = json.load(file)

chunks = [item['content'] for item in data]
embeddings = embedding_model.encode(chunks).astype('float32')

# # FAISS Index
# dimension = embeddings.shape[1]
# index = faiss.IndexFlatIP(dimension)
# index.add(embeddings)

nlist = 100  # Number of clusters (buckets)
quantizer = faiss.IndexFlatL2(embeddings.shape[1])
index = faiss.IndexIVFFlat(quantizer, embeddings.shape[1], nlist)

index.train(embeddings) # IVF requires training to find the clusters
index.add(embeddings)
index.nprobe = 10 # How many nearby buckets to check during search

In [ ]:
## Search Function
def retrieve_expert_data(query, top_k = 5):
  query_vector = embedding_model.encode([query]).astype('float32')
  D, I = index.search(query_vector, top_k)

  results = []
  for idx in I[0]:
    results.append(data[idx])
  return results

In [ ]:
query = "How does the slender loris avoid predators at night?"
query1 = "which species are called finger monkey ??"
matches = retrieve_expert_data(query1)

for m in matches:
    print(f"Topic: {m['subject']}\n{m['topic']}\nContent: {m['content']}\n")

Topic: Pygmy Marmoset
Genus
Content: Pygmy marmosets are the smallest true monkey, with a head-body length ranging from 117 to 152 mm (4.6 to 6.0 in) and a tail of 172 to 229 mm (6.8 to 9.0 in).

Topic: Macaque
Genus
Content: See text
 The macaques (/məˈkɑːk, -ˈkæk/) constitute a genus (Macaca) of gregarious Old World monkeys of the subfamily Cercopithecinae.

Topic: Pygmy Marmoset
Genus
Content: Pygmy marmosets are two species of small New World monkeys in the genus Cebuella.

Topic: Macaque
Gpe
Content: Aside from humans (genus Homo), the macaques are the most widespread primate genus, ranging from Japan to the Indian subcontinent, and in the case of the Barbary macaque (Macaca sylvanus), to North Africa and Southern Europe.

Topic: Macaque
Genus
Content: Although several species lack tails, and their common names refer to them as apes, these are true monkeys, with no greater relationship to the true apes than any other Old World monkeys.

